# Chapter 4: Agent Integration Patterns

Connecting autonomous agents into a functional team requires **explicit contracts**.
Agents must declare their identity, discover peers, and be observable to operators.

**Concepts covered:**
- WellDefinedAgent Protocol (`validate_scope`)
- Agent Registry with semantic discovery
- Anti-Corruption Layer (ACL) pattern
- Health checking and liveness probes

In [ ]:
import sys, os, asyncio
sys.path.insert(0, os.path.abspath('../..'))

from core.agents.base import AgentCapability, BaseAgent, CognitiveDomain, ResourceConstraints
from core.agents.registry import AgentProfile, AgentRegistry
print('Agent integration module ready.')

## 4.1 Well-Defined Agent Protocol

Every agent must satisfy the `WellDefinedAgent` protocol — it's the equivalent of
an API contract but for autonomous actors.

Key enforcement: **`validate_scope(task)`** → returns `False` for tasks outside the agent's domain,
preventing domain leakage before any LLM call is made.

In [ ]:
import asyncio

# Inventory agent with restricted scope
inventory_agent = BaseAgent(
    cognitive_domain=CognitiveDomain(
        name='InventoryManagement',
        description='Manages product stock levels and warehouse operations',
        tools=['check_stock', 'reserve_items', 'update_inventory'],
    ),
    capabilities=frozenset({
        AgentCapability.READ_DATA,
        AgentCapability.WRITE_DATA,
        AgentCapability.EMIT_EVENTS,
    }),
)

async def test_scope():
    test_cases = [
        ('check_stock for SKU-42', True),
        ('reserve_items for order ORD-123', True),
        ('process credit card payment for $99', False),
        ('delete all customer records', False),
        ('update_inventory after shipment received', True),
    ]
    print('Scope validation results:')
    for task, expected in test_cases:
        in_scope = await inventory_agent.validate_scope(task)
        status = '✓' if in_scope == expected else '✗'
        print(f'  {status} [{"IN" if in_scope else "OUT"}] {task[:50]}')

asyncio.run(test_scope())

## 4.2 Agent Registry and Semantic Discovery

The registry solves **capability discovery** without hardcoded routing tables.
Agents register their profile; callers query with a natural-language task description.

Production: embed using **Azure OpenAI `text-embedding-3-small`** for true semantic matching.
Demo: uses SHA-256 stub embeddings (deterministic, zero-cost).

In [ ]:
# Create a registry with multiple specialized agents
registry = AgentRegistry()

agents = [
    AgentProfile(
        agent_id='order-agent-001',
        name='OrderProcessingAgent',
        description='Processes customer orders, validates payment, triggers fulfillment',
        capabilities=['process_order', 'validate_payment', 'check_fraud'],
        endpoint='http://order-agent:8080',
    ),
    AgentProfile(
        agent_id='inventory-agent-001',
        name='InventoryAgent',
        description='Manages product stock, reservations, and warehouse operations',
        capabilities=['check_stock', 'reserve_items', 'update_inventory'],
        endpoint='http://inventory-agent:8080',
    ),
    AgentProfile(
        agent_id='notification-agent-001',
        name='NotificationAgent',
        description='Sends emails, SMS, and push notifications to customers',
        capabilities=['send_email', 'send_sms', 'send_push'],
        endpoint='http://notification-agent:8080',
    ),
    AgentProfile(
        agent_id='analytics-agent-001',
        name='AnalyticsAgent',
        description='Tracks events, builds dashboards, and generates business reports',
        capabilities=['track_event', 'query_metrics', 'generate_report'],
        endpoint='http://analytics-agent:8080',
    ),
]

for profile in agents:
    registry.register(profile)

print(f'Registered {len(registry.list_agents())} agents\n')

# Semantic task routing
tasks = [
    'Reserve 3 units of SKU-789 for new order',
    'Send order confirmation email to customer@example.com',
    'Generate weekly revenue report for dashboard',
]

for task in tasks:
    match = registry.find_best_match(task)
    print(f'Task: {task[:55]}')
    print(f'  → Routed to: {match.name} (id: {match.agent_id[:12]}...)\n')

## 4.3 Anti-Corruption Layer (ACL)

When two bounded contexts must communicate, we need a **translation layer** that
prevents the language and concepts of one domain from corrupting the other.

Example: the legacy CRM uses `client_code` while the new order system uses `customer_id`.
The ACL translates between them transparently.

In [ ]:
# Anti-Corruption Layer: translates between legacy CRM and new order domain
class CRMAntiCorruptionLayer:
    """Translates legacy CRM concepts to Order domain language."""

    # CRM field names → Order domain field names
    FIELD_MAP = {
        'client_code': 'customer_id',
        'order_ref': 'order_id',
        'ship_address': 'delivery_address',
        'item_codes': 'line_items',
        'order_value': 'total_amount',
    }

    def translate_inbound(self, crm_payload: dict) -> dict:
        """CRM → Order domain"""
        return {
            self.FIELD_MAP.get(k, k): v
            for k, v in crm_payload.items()
        }

    def translate_outbound(self, order_payload: dict) -> dict:
        """Order domain → CRM"""
        reverse_map = {v: k for k, v in self.FIELD_MAP.items()}
        return {reverse_map.get(k, k): v for k, v in order_payload.items()}

# Demo
acl = CRMAntiCorruptionLayer()

crm_event = {
    'client_code': 'CRM-4892',
    'order_ref': 'REF-2024-001',
    'ship_address': '123 Main St, London',
    'item_codes': ['SKU-001', 'SKU-042'],
    'order_value': 149.99,
}

translated = acl.translate_inbound(crm_event)
print('Original CRM payload:', crm_event)
print()
print('Translated (Order domain):', translated)

## 4.4 Health Checking

Every agent exposed in a microservices environment must implement:
- `/health/live` – liveness: is the process running?
- `/health/ready` – readiness: is the agent ready to accept work?

Azure Container Apps uses these probes to route traffic and restart unhealthy replicas.

In [ ]:
from dataclasses import dataclass
from enum import auto, Enum

class HealthStatus(Enum):
    HEALTHY = auto()
    DEGRADED = auto()
    UNHEALTHY = auto()

@dataclass
class HealthCheck:
    name: str
    status: HealthStatus
    message: str

@dataclass
class AgentHealth:
    agent_id: str
    checks: list[HealthCheck]

    @property
    def is_live(self) -> bool:
        return any(c.status != HealthStatus.UNHEALTHY for c in self.checks)

    @property
    def is_ready(self) -> bool:
        return all(c.status == HealthStatus.HEALTHY for c in self.checks)

    def report(self) -> dict:
        return {
            'agent_id': self.agent_id,
            'live': self.is_live,
            'ready': self.is_ready,
            'checks': [{'name': c.name, 'status': c.status.name, 'detail': c.message}
                        for c in self.checks],
        }

# Simulate a health report
import json
health = AgentHealth(
    agent_id='order-agent-001',
    checks=[
        HealthCheck('database', HealthStatus.HEALTHY, 'Connected to orders-db'),
        HealthCheck('llm_api', HealthStatus.HEALTHY, 'Azure OpenAI responding in <200ms'),
        HealthCheck('event_bus', HealthStatus.DEGRADED, 'Service Bus lag > 1s'),
    ],
)

report = health.report()
print(json.dumps(report, indent=2))
print(f'\nReady for traffic: {report["ready"]}')